# 8장. 작은 데이터 분석 프로젝트 완성하기

강사 공식 Notebook `notebooks/ch08_midterm_project.ipynb`의 실행 흐름을 유지하고, `practice/chapter08/chapter08.md`와 `practice/chapter08/templates/chapter08_assignment.md`의 제출 형식에 맞춰 답안 Markdown 셀을 추가한 제출용 Notebook입니다.

> 실행 성공만으로 완료 처리하지 않고, PK/FK·병합·계산식·총합·날짜·개인정보·최종 Validation을 함께 확인합니다.


## 제출 정보
- 이름: 조영우
- GitHub ID: `cyw0927`
- 작성일: 2026-09-17
- 최종 Notebook URL: `https://github.com/cyw0927/-llm-data-analysis-course/blob/main/assignments/chapter08/chapter08.ipynb`


## 1. 프로젝트 질문
### 분석 질문
1. 카테고리별 completed 주문 기준 금액은 어떻게 다른가?
2. 월별 completed 주문 기준 금액과 주문 수는 어떻게 변하는가?
3. completed 주문 기준 고객별 구매 금액에는 어떤 차이가 있는가?

### 분석 범위
- 금액성 분석은 `order_status == "completed"` 주문만 사용한다.
- 주문 수는 주문 상세 행 수가 아니라 `order_id.nunique()` 기준으로 계산한다.
- 원본 CSV는 직접 수정하지 않고 전처리된 복사본을 사용한다.

### 사용할 데이터와 지표
- `data/raw/customers.csv`: 고객 정보
- `data/raw/products.csv`: 상품·카테고리·가격 정보
- `data/raw/orders.csv`: 주문일·주문상태·고객 연결
- `data/raw/order_items.csv`: 수량·단가·상품 연결
- 주요 지표: `line_total`, completed 주문 금액, 주문 수, 고객별 구매 금액

### 계산 기준
- `order_status == "completed"` 적용 여부: 적용
- `line_total = quantity × unit_price` 확인 여부: 아래 검증 셀에서 확인

### 완료 기준
PK/FK, 병합, line_total, category/month/customer 총합, 날짜, 개인정보, 최종 Validation이 모두 정상이고 전체 파이프라인이 재실행 가능해야 한다.


### 프로젝트 루트와 출력 폴더 설정


In [ ]:
from pathlib import Path
import sys
import shutil

import numpy as np
import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트 폴더를 찾을 수 없습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
ASSIGNMENT_IMAGE_DIR = PROJECT_ROOT / "assignments" / "chapter08" / "images"

for path in [PROCESSED_DIR, REPORT_DIR, FIGURE_DIR, ASSIGNMENT_IMAGE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python 실행 파일:", sys.executable)
print("프로젝트 루트:", PROJECT_ROOT)
print("원본 데이터 폴더:", RAW_DIR)
print("보고서 폴더:", REPORT_DIR)


### 프로젝트 함수 불러오기


In [ ]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)
from src.midterm_project import (
    FORBIDDEN_CUSTOMER_COLUMNS,
    build_analysis_tables,
    build_interpretation_notes,
    build_key_duplicate_checks,
    build_midterm_report,
    build_project_validation,
    create_project_figures,
    run_midterm_project,
    save_project_tables,
    summarize_datasets,
)


## 2. 입력 데이터와 전처리 검증
- 사용 원본 파일: `customers.csv`, `products.csv`, `orders.csv`, `order_items.csv`
- shape: 아래 `dataset_summary`와 각 DataFrame 출력에서 확인
- 핵심 결측/중복 결과: 아래 원본 점검 출력에서 확인
- 전처리 전후 변화: `preprocessing_comparison`에서 확인


In [ ]:
raw_data = load_sales_data(RAW_DIR)

dataset_summary = summarize_datasets(raw_data)
display(dataset_summary)

for name, df in raw_data.items():
    print(f"\n===== {name} =====")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("missing values:", int(df.isna().sum().sum()))
    print("duplicated rows:", int(df.duplicated().sum()))


In [ ]:
processed_data = preprocess_sales_data(raw_data)
processed_paths = save_processed_data(processed_data, PROCESSED_DIR)

preprocessing_comparison = compare_shapes(raw_data, processed_data)
display(preprocessing_comparison)

for path in processed_paths:
    print(path, "OK" if path.exists() else "MISSING")


### 나의 해석과 판단
원본 데이터 구조와 전처리 전후 행 수를 먼저 확인해야 이후 집계값의 변화를 설명할 수 있다고 판단했다. 원본은 보존하고 분석용 복사본을 만드는 방식이 재현성과 오류 추적에 유리하다.

### 한계와 추가 확인 사항
전처리 과정에서 행이 제거되거나 형 변환이 실패한다면 정보 손실이 생길 수 있으므로 단순 실행 성공만 보지 않고 전후 shape와 변환 실패를 함께 확인해야 한다.

![입력 데이터 검증](images/step02_data_validation.png)


## 3. PK/FK·병합 검증
### PK 결과
- 결측: 아래 `key_duplicate_checks`의 `missing_count` 확인
- 중복: 아래 `duplicate_count` 확인
- PASS/FAIL: `status` 확인

### FK 결과
- 미매칭: 아래 `relationship_checks.invalid_count` 확인
- PASS/FAIL: `status` 확인


In [ ]:
key_duplicate_checks = build_key_duplicate_checks(processed_data)
relationship_checks = validate_relationships(processed_data).copy()

if not relationship_checks.empty:
    relationship_checks["status"] = (
        relationship_checks["invalid_count"]
        .eq(0)
        .map({True: "PASS", False: "FAIL"})
    )

display(key_duplicate_checks)
display(relationship_checks)

assert key_duplicate_checks["status"].eq("PASS").all(), "PK Validation 실패"
assert (
    not relationship_checks.empty
    and relationship_checks["invalid_count"].eq(0).all()
), "FK Validation 실패"

print("PK/FK Gate: PASS")


In [ ]:
analysis_tables = build_analysis_tables(processed_data)

display(analysis_tables["merge_checks"])
display(analysis_tables["line_total_check"])
display(analysis_tables["date_checks"])
display(analysis_tables["amount_scope_summary"])

assert analysis_tables["merge_checks"]["status"].eq("PASS").all()
assert analysis_tables["line_total_check"]["status"].eq("PASS").all()

print("병합 / line_total Gate: PASS")


### 병합 결과
- validate 관계: 주문상세→주문 `many_to_one`, 주문상세→상품 `many_to_one`, 고객집계→고객 `one_to_one`
- 병합 전 행 수: `merge_checks`의 before rows 확인
- 병합 후 행 수: `merge_checks`의 after rows 확인
- 미매칭: `merge_checks`의 unmatched 확인
- PASS/FAIL: `merge_checks.status` 확인

### 나의 해석과 판단
`left merge`라도 오른쪽 키가 중복이면 행이 늘어날 수 있으므로, 관계 선언과 행 수·미매칭을 함께 확인해야 한다. `line_total`도 기존 컬럼을 그대로 믿지 않고 `quantity × unit_price`와 다시 대조한다.


## 4. 핵심 EDA 결과


In [ ]:
category_sales = analysis_tables["category_sales"]
monthly_sales = analysis_tables["monthly_sales"]
customer_sales_internal = analysis_tables["customer_sales"]
customer_sales_public = analysis_tables["customer_sales_public"]
order_status_summary = analysis_tables["order_status_summary"]

display(category_sales)
display(monthly_sales)
display(customer_sales_public.head(10))
display(order_status_summary)

print("내부 고객 결과 컬럼:", customer_sales_internal.columns.tolist())
print("공개 고객 결과 컬럼:", customer_sales_public.columns.tolist())


### 결과 1 — 카테고리별 completed 주문 기준 금액
- 수치/표: 바로 위 `category_sales` 출력 사용
- 결과 관찰: 기존 실습에서는 스포츠 31,743,000, 전자기기 26,400,000으로 확인했다. 이번 공식 프로젝트 결과와 다시 대조한다.
- 나의 해석과 판단: 금액 차이만으로 선호도 차이라고 단정하지 않는다.
- 업무·분석적 의미: 판매 수량과 평균 단가를 분리하면 차이의 구조를 더 잘 볼 수 있다.
- 한계: 프로모션·재고·계절성 같은 원인 데이터가 없다.

### 결과 2 — 월별 completed 주문 기준 금액과 주문 수
- 수치/표: 바로 위 `monthly_sales` 출력 사용
- 결과 관찰: 기존 실습에서는 2025-07 5,869,000/8건, 2025-08 15,621,000/18건이었다. 이번 공식 프로젝트 결과와 다시 대조한다.
- 나의 해석과 판단: 증가 자체는 관찰할 수 있지만 원인은 현재 데이터만으로 특정할 수 없다.
- 업무·분석적 의미: 주문 수와 평균 주문 금액을 분리해 변화 구조를 확인할 필요가 있다.
- 한계: 광고·할인·시즌 효과 같은 설명 변수가 없다.

### 결과 3 — 고객별 completed 주문 구매 금액
- 수치/표: `customer_sales_public.head(10)` 출력 사용
- 결과 관찰: 고객별 completed 구매 금액과 주문 횟수에 차이가 있다.
- 나의 해석과 판단: 고액 구매 고객과 반복 구매 고객은 같은 의미가 아니므로 총금액·주문 수·평균 주문금액을 함께 본다.
- 업무·분석적 의미: 고객 세분화나 추가 분석 질문으로 연결할 수 있다.
- 한계: 현재 결과만으로 충성도나 이탈 가능성을 단정할 수 없다.


## 5. Total consistency와 날짜 검증


In [ ]:
total_consistency = analysis_tables["total_consistency_check"]
display(total_consistency)
display(analysis_tables["date_checks"])

assert total_consistency["matches_completed"].all(), "Total consistency Gate 실패"
assert analysis_tables["date_checks"]["status"].eq("PASS").all(), "날짜 Gate 실패"

print("Total consistency / Date Gate: PASS")


- completed source total: 바로 위 `total_consistency`의 기준값 확인
- category total: `category_completed_amount` 행 확인
- monthly total: `monthly_completed_amount` 행 확인
- customer total: `customer_completed_amount` 행 확인
- 총합 일치 여부: `matches_completed == True` 확인
- completed 주문 날짜 오류 건수: `date_checks` 확인
- 날짜 오류 영향 금액: `date_checks` 확인

### 불일치가 있었다면 원인
불일치가 생기면 completed 필터 누락, 병합 행 증식, FK 미매칭, category 결측, 날짜 변환 실패, 고객 연결 누락, 서로 다른 실행 시점의 DataFrame 사용 여부를 순서대로 확인한다.


## 6. 대표 시각화


In [ ]:
saved_figures = create_project_figures(
    analysis_tables,
    FIGURE_DIR,
    show=True,
)

for path in saved_figures:
    print(path.name, path.exists(), path.stat().st_size)

# 제출용 images 폴더에도 대표 그래프 2개를 복사한다.
figure_map = {
    FIGURE_DIR / "ch08_category_sales.png": ASSIGNMENT_IMAGE_DIR / "graph01.png",
    FIGURE_DIR / "ch08_monthly_sales.png": ASSIGNMENT_IMAGE_DIR / "graph02.png",
}
for src, dst in figure_map.items():
    if src.exists():
        shutil.copy2(src, dst)
        print("copied:", dst)


![대표 그래프 1](images/graph01.png)
![대표 그래프 2](images/graph02.png)

### 그래프 선택 이유
카테고리 비교는 막대그래프가 항목 간 크기 차이를 보기 쉽고, 월별 변화는 선그래프가 시간 흐름을 확인하기 적합하다.

### 그래프와 원본 집계값 일치 여부
그래프는 별도의 새 계산이 아니라 검증된 `category_sales`, `monthly_sales` 집계표에서 생성한다.

### 그래프에서 직접 관찰한 사실
카테고리별 completed 주문 금액과 월별 completed 주문 금액이 동일하지 않고 구간별 차이가 존재한다.

### 그래프만으로 말할 수 없는 것
프로모션 성공 여부, 고객 선호, 수익성, 계절성 같은 원인은 현재 그래프만으로 단정할 수 없다.


## 7. 개인정보 검증


In [ ]:
forbidden = sorted(
    FORBIDDEN_CUSTOMER_COLUMNS.intersection(customer_sales_public.columns)
)
print("공개 결과 금지 컬럼:", forbidden)

assert not forbidden, "Privacy Gate 실패"
assert "customer_id" not in customer_sales_public.columns
print("Privacy Gate: PASS")


In [ ]:
interpretation_notes = build_interpretation_notes()
display(interpretation_notes)

saved_tables = save_project_tables(
    dataset_summary,
    preprocessing_comparison,
    key_duplicate_checks,
    relationship_checks,
    analysis_tables,
    interpretation_notes,
    REPORT_DIR,
)

for path in saved_tables:
    print(path.name, path.exists(), path.stat().st_size)


In [ ]:
saved_customer = pd.read_csv(REPORT_DIR / "ch08_customer_sales.csv")

saved_forbidden = sorted(
    FORBIDDEN_CUSTOMER_COLUMNS.intersection(saved_customer.columns)
)
print("저장 CSV 컬럼:", saved_customer.columns.tolist())
print("금지 컬럼:", saved_forbidden)

assert not saved_forbidden
print("저장된 고객 CSV Privacy Gate: PASS")


- 공개 고객 CSV 컬럼: 바로 위 출력에서 확인
- 원본 `customer_id` 포함 여부: 포함하지 않음이 PASS 조건
- 이름/이메일/전화번호/주소 포함 여부: 포함하지 않음이 PASS 조건
- 익명 라벨 방식: `Customer 01`, `Customer 02` 같은 순위 기반 라벨
- PASS/FAIL: Privacy Gate 출력 확인

### 나의 판단
분석 내부에서는 식별 키가 필요할 수 있지만 공개 산출물에는 직접 식별정보를 남길 이유가 없으므로 내부용 키와 공개용 결과를 분리하는 것이 적절하다.


## 8. 최종 Validation


In [ ]:
project_validation = build_project_validation(
    key_duplicate_checks,
    relationship_checks,
    analysis_tables,
)
display(project_validation)

assert project_validation["status"].eq("PASS").all(), "최종 Validation 실패"
print("Project Validation: PASS")


`reports/ch08_project_validation.csv`의 결과를 요약합니다.

| 검증 항목 | 결과 | 내가 확인한 근거 |
| --- | --- | --- |
| pk_integrity | 위 실행 결과 확인 | PK 결측/중복 검사 |
| fk_integrity | 위 실행 결과 확인 | FK 미매칭 검사 |
| merge_checks_pass | 위 실행 결과 확인 | 병합 전후 행 수·미매칭 검사 |
| line_total_consistency | 위 실행 결과 확인 | `quantity × unit_price` 대조 |
| completed_total_consistency | 위 실행 결과 확인 | category/month/customer 총합 대조 |
| category_sales_ratio_pct_sum | 위 실행 결과 확인 | 카테고리 비율 합계 검사 |
| completed_rows_with_invalid_order_date | 위 실행 결과 확인 | completed 날짜 오류 검사 |
| public_customer_columns_safe | 위 실행 결과 확인 | 공개 고객 컬럼 검사 |

### FAIL이 있었다면 수정 내용
FAIL이 있으면 원인을 수정한 뒤 Notebook과 전체 스크립트를 처음부터 다시 실행한다.


## 9. LLM 활용 기록
- 사용 여부: 예
- 사용 목적: 코드 설명, 과제 요구사항 정리, 검증 항목 점검, 문장 표현 보조
- Safe Context: 데이터 구조, 코드, 익명 집계 결과, 검증 결과만 사용
- Prompt 요약: Chapter 08 공식 Notebook과 실습 가이드·답안 템플릿을 비교해 누락된 검증과 제출 항목을 확인
- 제안 요약: 공식 실행 코드는 유지하고 템플릿의 1~11 답안 구조를 Markdown 셀로 추가
- 반영/수정/보류: 실제 실행 결과와 일치하는 내용만 반영하고 데이터에 없는 원인은 단정하지 않음
- 사람이 검증한 근거: Notebook의 실제 출력, CSV Evidence, 최종 Validation을 기준으로 확인


### LLM 검토 프롬프트 예시
```text
온라인 쇼핑몰 중간 프로젝트 결과를 검토해 주세요.
계산 기준은 line_total = quantity × unit_price이며 금액성 분석은 completed 주문만 사용합니다.
PK/FK, merge, total consistency, 날짜, 개인정보 Validation 결과를 기준으로 과도한 해석이나 추가 검증 항목을 점검해 주세요.
데이터에 없는 원인은 추정하지 마세요.
```


## 10. 프로젝트 재현 확인
- `python scripts/run_midterm_project.py` 실행 여부: 아래 재실행 셀/터미널 실행으로 확인
- 재실행 결과: 최종 Validation 표에서 확인
- Notebook과 핵심 수치 일치 여부: completed source/category/month/customer 총합을 비교
- 생성된 핵심 Evidence 파일 확인 여부: 아래 산출물 검증에서 확인


In [ ]:
report_text = build_midterm_report(
    dataset_summary,
    preprocessing_comparison,
    key_duplicate_checks,
    relationship_checks,
    analysis_tables,
    interpretation_notes,
)

report_path = REPORT_DIR / "ch08_midterm_report.md"
report_path.write_text(report_text, encoding="utf-8")
print("보고서 저장:", report_path)


In [ ]:
project_result = run_midterm_project(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    figure_dir=FIGURE_DIR,
    show_figures=False,
)

display(project_result["project_validation"])
project_result["report_path"]


In [ ]:
expected_outputs = [
    REPORT_DIR / "ch08_midterm_report.md",
    REPORT_DIR / "ch08_dataset_summary.csv",
    REPORT_DIR / "ch08_preprocessing_comparison.csv",
    REPORT_DIR / "ch08_key_duplicate_checks.csv",
    REPORT_DIR / "ch08_relationship_checks.csv",
    REPORT_DIR / "ch08_merge_checks.csv",
    REPORT_DIR / "ch08_line_total_check.csv",
    REPORT_DIR / "ch08_date_checks.csv",
    REPORT_DIR / "ch08_amount_scope_summary.csv",
    REPORT_DIR / "ch08_total_consistency_check.csv",
    REPORT_DIR / "ch08_project_validation.csv",
    REPORT_DIR / "ch08_category_sales.csv",
    REPORT_DIR / "ch08_monthly_sales.csv",
    REPORT_DIR / "ch08_customer_sales.csv",
    REPORT_DIR / "ch08_order_status_summary.csv",
    REPORT_DIR / "ch08_interpretation_notes.csv",
    FIGURE_DIR / "ch08_category_sales.png",
    FIGURE_DIR / "ch08_monthly_sales.png",
    FIGURE_DIR / "ch08_top_customers.png",
]

all_outputs_valid = True
for path in expected_outputs:
    valid = path.exists() and path.stat().st_size > 0
    all_outputs_valid &= valid
    print(path.name, "OK" if valid else "MISSING OR EMPTY")

assert all_outputs_valid
print("전체 산출물 검증: PASS")


![재실행 결과](images/step06_reproduce.png)

### 나의 해석과 판단
재실행 가능한 프로젝트는 한 번 우연히 동작한 코드가 아니라 같은 원본과 같은 기준에서 다시 실행했을 때 같은 결과를 얻을 수 있어야 한다. 따라서 코드뿐 아니라 필터 기준, 병합 관계, 계산식, Validation과 산출물까지 함께 남기는 것이 중요하다.


## 11. 최종 프로젝트 요약
### 핵심 인사이트 3개
1. completed 주문 기준 카테고리별 금액에는 차이가 있으며, 기존 실습에서 스포츠와 전자기기 금액이 크게 나타났다. 공식 프로젝트 실행 결과와 최종 대조한다.
2. 월별 completed 주문 금액과 주문 수는 기간별 차이가 있었으며, 기존 실습에서는 2025년 7월보다 8월 값이 높았다.
3. 고객별 구매 금액은 고객마다 차이가 있으므로 총금액뿐 아니라 주문 횟수와 평균 주문 금액을 함께 봐야 한다.

### 가장 중요한 업무·분석적 의미
큰 숫자를 찾는 것보다 동일한 분석 범위와 계산 기준을 유지하고 서로 다른 집계 결과를 교차 검증하는 과정이 중요하다.

### 현재 분석의 한계
- 프로모션, 광고, 재고, 계절성 데이터가 없다.
- completed 주문 기준 금액을 회계상 순매출이라고 단정할 수 없다.
- 집계 결과만으로 고객 행동의 원인을 설명할 수 없다.

### 다음 분석 제안
1. 카테고리별 판매 수량과 평균 단가를 분리해 비교
2. 월별 주문 수와 평균 주문 금액을 분리해 비교
3. 고객별 최근 구매일·구매 빈도·평균 구매 금액을 추가 분석


## 최종 체크
- [ ] 질문과 지표가 연결됩니다.
- [ ] 원본 데이터에서 다시 시작했습니다.
- [ ] PK/FK·병합 검증 근거가 있습니다.
- [ ] `line_total` 계산 관계를 확인했습니다.
- [ ] 금액성 분석은 completed 범위를 사용했습니다.
- [ ] category/month/customer 총합이 source total과 일치합니다.
- [ ] 날짜 오류를 확인했습니다.
- [ ] 공개 고객 결과에서 원본 ID와 직접 식별정보를 제거했습니다.
- [ ] 대표 그래프와 원본 집계값을 비교했습니다.
- [ ] 원인 과대 해석이 없습니다.
- [ ] 최종 Validation이 모두 PASS입니다.
- [ ] 전체 프로젝트를 재실행했습니다.
- [ ] 한계와 다음 분석을 작성했습니다.
- [ ] 최종 Notebook URL을 제출합니다.

> 위 체크는 `Run All` 및 `python scripts/run_midterm_project.py` 재실행 후 실제 결과를 확인한 다음 `[x]`로 변경합니다.
